In [3]:
import numpy as np, pandas as pd
 
D = 0.008   # d = tasa de descuento mensual (0.8%)
 
datos = pd.DataFrame({
    'segmento': ['Constantes', 'Casuales'],
    'precio':   [20, 12],      # P = precio de hoy
    'costo':    [8, 8],        # C = costo por cliente
    'unidades': [1000, 2000],  # Q = unidades al mes hoy
    'eps_x':    [0.5, 0.2],    # εx = efecto de SU precio en las unidades del OTRO segmento
    'churn':    [0.03, 0.06],  # c = churn mensual
}).set_index('segmento')
datos


,precio,costo,unidades,eps_x,churn
segmento,,,,,
Constantes,20,8,1000,0.5,0.03
Casuales,12,8,2000,0.2,0.06


In [2]:
# Pruebas de precio anteriores: a cada precio, cuántas unidades se vendieron al mes
pruebas = pd.DataFrame({
    'segmento': ['Constantes'] * 5 + ['Casuales'] * 5,
    'precio': [18, 19, 20, 21, 22,
               10.8, 11.4, 12, 12.6, 13.2],
    'unidades': [1159, 1074, 1000, 934, 875,
                 2603, 2274, 2000, 1770, 1576]
})

pruebas

,segmento,precio,unidades
0,Constantes,18.0,1159
1,Constantes,19.0,1074
2,Constantes,20.0,1000
3,Constantes,21.0,934
4,Constantes,22.0,875
5,Casuales,10.8,2603
6,Casuales,11.4,2274
7,Casuales,12.0,2000
8,Casuales,12.6,1770
9,Casuales,13.2,1576


In [4]:
# EJEMPLO resuelto: elasticidad de Constantes
g = pruebas[pruebas.segmento == 'Constantes']
 
# Pendiente de la recta ln(unidades) contra ln(precio): es la elasticidad ε (como en el Demo 1)
eps_const = np.polyfit(np.log(g.precio), np.log(g.unidades), 1)[0]
print(f'Elasticidad de Constantes: {eps_const:.2f}')


Elasticidad de Constantes: -1.40


In [5]:

# TU TURNO: elasticidad de Casuales
g = pruebas[pruebas.segmento == 'Casuales']

# Calcular la elasticidad de Casuales
eps_casu = np.polyfit(np.log(g.precio), np.log(g.unidades), 1)[0]

# Mostrar el resultado
print(f'Elasticidad de Casuales: {eps_casu:.2f}')

# Agregar las elasticidades al DataFrame
datos['eps'] = [round(eps_const, 1), round(eps_casu, 1)]

# Mostrar los datos actualizados
datos


Elasticidad de Casuales: -2.50


,precio,costo,unidades,eps_x,churn,eps
segmento,,,,,,
Constantes,20,8,1000,0.5,0.03,-1.4
Casuales,12,8,2000,0.2,0.06,-2.5


In [6]:
# Las mismas funciones del Demo 2 de Ritmo (no hay que cambiarlas)
def precio_optimo(C, eps):
    return C * eps / (1 + eps)                 # P* = C × ε ÷ (1 + ε)
 
def aplicar_reglas(p_hoy, p_opt, C):
    p = min(p_opt, p_hoy * 1.10)               # tope: sube máximo 10%
    p = max(p, p_hoy * 0.90)                   # tope: baja máximo 10%
    p = max(p, 1.2 * C)                        # margen mínimo: precio ≥ 1.2 × C
    return p
 
def utilidad(p, p_hoy, C, Q, eps):
    unidades = Q * (p / p_hoy) ** eps          # cuántas unidades se venden a p
    return (p - C) * unidades                  # utilidad = (p − C) × unidades
 
def clv(m, c, d=D):
    return m * (1 - c) / (c + d)               # CLV = m × (1 − c) ÷ (c + d)
 
def churn_maximo(m_nuevo, clv_hoy, d=D):
    return (m_nuevo - clv_hoy * d) / (m_nuevo + clv_hoy)   # c'


In [7]:
# EJEMPLO resuelto: segmento Constantes
s = datos.loc['Constantes']
 
p_opt  = precio_optimo(s.costo, s.eps)
p_prop = aplicar_reglas(s.precio, p_opt, s.costo)
 
print(f'Precio óptimo P*:          ${p_opt:.2f}')
print(f'Precio propuesto (reglas): ${p_prop:.2f}')
print(f'Utilidad hoy:      {utilidad(s.precio, s.precio, s.costo, s.unidades, s.eps):,.0f}')
print(f'Utilidad propuesta:{utilidad(p_prop, s.precio, s.costo, s.unidades, s.eps):,.0f}')


Precio óptimo P*:          $28.00
Precio propuesto (reglas): $22.00
Utilidad hoy:      12,000
Utilidad propuesta:12,251


In [8]:
#CALCULO PREGUNNTA #3

# Calcular utilidad con el precio óptimo de $28
utilidad_optima = utilidad(p_opt, s.precio, s.costo, s.unidades, s.eps)

# Calcular utilidad con el precio permitido de $22
utilidad_propuesta = utilidad(p_prop, s.precio, s.costo, s.unidades, s.eps)

# Diferencia por la restricción
diferencia = utilidad_optima - utilidad_propuesta

print(f'Utilidad con precio óptimo: ${utilidad_optima:,.2f}')
print(f'Utilidad con precio propuesto: ${utilidad_propuesta:,.2f}')
print(f'Utilidad que cuesta la restricción: ${diferencia:,.2f}')


Utilidad con precio óptimo: $12,486.79
Utilidad con precio propuesto: $12,251.19
Utilidad que cuesta la restricción: $235.60


In [9]:

# TU TURNO: segmento Casuales
s = datos.loc['Casuales']

# Calcular precio óptimo y precio propuesto
p_opt = precio_optimo(s.costo, s.eps)
p_prop = aplicar_reglas(s.precio, p_opt, s.costo)

# Calcular utilidades
utilidad_hoy = utilidad(s.precio, s.precio, s.costo, s.unidades, s.eps)
utilidad_propuesta = utilidad(p_prop, s.precio, s.costo, s.unidades, s.eps)
utilidad_optima = utilidad(p_opt, s.precio, s.costo, s.unidades, s.eps)

# Calcular diferencias
incremento = utilidad_propuesta - utilidad_hoy
costo_restriccion = utilidad_optima - utilidad_propuesta

# Mostrar resultados
print(f'Precio óptimo P*:           ${p_opt:.2f}')
print(f'Precio propuesto (reglas):  ${p_prop:.2f}')
print(f'Utilidad hoy:              ${utilidad_hoy:,.2f}')
print(f'Utilidad propuesta:        ${utilidad_propuesta:,.2f}')
print(f'Utilidad óptima:           ${utilidad_optima:,.2f}')
print(f'Incremento de utilidad:    ${incremento:,.2f}')
print(f'Costo de la restricción:   ${costo_restriccion:,.2f}')


Precio óptimo P*:           $13.33
Precio propuesto (reglas):  $13.20
Utilidad hoy:              $8,000.00
Utilidad propuesta:        $8,195.05
Utilidad óptima:           $8,196.62
Incremento de utilidad:    $195.05
Costo de la restricción:   $1.57


In [10]:
# La función nueva: utilidad de los DOS segmentos juntos, con elasticidad cruzada
def utilidad_total(p_const, p_casu):
    a, b = datos.loc['Constantes'], datos.loc['Casuales']
    q_const = a.unidades * (p_const / a.precio) ** a.eps * (p_casu / b.precio) ** b.eps_x
    q_casu  = b.unidades * (p_casu / b.precio) ** b.eps * (p_const / a.precio) ** a.eps_x
    u_const = (p_const - a.costo) * q_const
    u_casu  = (p_casu - b.costo) * q_casu
    return u_const, u_casu
 
# EJEMPLO resuelto: sube SOLO Constantes a $22 (Casuales se queda en $12)
hoy = utilidad_total(20, 12)
sube = utilidad_total(22, 12)
print(f'Constantes: {hoy[0]:,.0f} -> {sube[0]:,.0f}')
print(f'Casuales:   {hoy[1]:,.0f} -> {sube[1]:,.0f}')
print(f'Total:      {sum(hoy):,.0f} -> {sum(sube):,.0f}')


Constantes: 12,000 -> 12,251
Casuales:   8,000 -> 8,390
Total:      20,000 -> 20,642


In [11]:

# TU TURNO: suben LOS DOS segmentos
# Constantes a $22 y Casuales a $13.20

# Calcular utilidad con ambos precios nuevos
u_const, u_casu = utilidad_total(22, 13.2)

# Calcular utilidad total
u_total = u_const + u_casu

# Mostrar resultados
print(f'Utilidad Constantes: ${u_const:,.2f}')
print(f'Utilidad Casuales:   ${u_casu:,.2f}')
print(f'Utilidad total:     ${u_total:,.2f}')

# Comparar con las utilidades sin efecto cruzado
u_sin_cruzado = 12251 + 8195

# Beneficio adicional de considerar el efecto cruzado
diferencia = u_total - u_sin_cruzado

print('\n--- Comparación ---')
print(f'Utilidad sin efecto cruzado: ${u_sin_cruzado:,.2f}')
print(f'Utilidad con efecto cruzado: ${u_total:,.2f}')
print(f'Ganancia adicional:         ${diferencia:,.2f}')


Utilidad Constantes: $12,486.96
Utilidad Casuales:   $8,595.04
Utilidad total:     $21,082.00

--- Comparación ---
Utilidad sin efecto cruzado: $20,446.00
Utilidad con efecto cruzado: $21,082.00
Ganancia adicional:         $636.00


In [12]:

# PARTE 5: CHURN MÁXIMO

# Precios propuestos
precios_propuestos = {
    'Constantes': 22,
    'Casuales': 13.20
}

resultados_churn = {}

# Calcular CLV actual y churn máximo
for segmento in ['Constantes', 'Casuales']:

    s = datos.loc[segmento]

    # 1) Margen y CLV actuales
    margen_hoy = s.precio - s.costo
    clv_hoy = clv(margen_hoy, s.churn)

    # 2) Margen con el precio nuevo
    m_nuevo = precios_propuestos[segmento] - s.costo

    # 3) Calcular churn máximo tolerable
    c_max = churn_maximo(m_nuevo, clv_hoy)

    resultados_churn[segmento] = c_max

    print(f'\n--- {segmento} ---')
    print(f'CLV actual: ${clv_hoy:.2f}')
    print(f'Margen nuevo: ${m_nuevo:.2f}')
    print(f'Churn actual: {s.churn:.2%}')
    print(f'Churn máximo: {c_max:.2%}')


# TABLA DE ESCENARIOS

escenarios = [0.5, 1, 2.5]
tabla = []

for e in escenarios:

    c_const = 0.03 * (1 + e * 0.10)
    c_casu = 0.06 * (1 + e * 0.10)

    tabla.append({
        'Elasticidad churn': e,
        'Churn Constantes': f'{c_const:.2%}',
        '¿Cabe Constantes?': 'Sí' if c_const <= resultados_churn['Constantes'] else 'No',
        'Churn Casuales': f'{c_casu:.2%}',
        '¿Cabe Casuales?': 'Sí' if c_casu <= resultados_churn['Casuales'] else 'No'
    })

tabla_churn = pd.DataFrame(tabla)
display(tabla_churn)



--- Constantes ---
CLV actual: $306.32
Margen nuevo: $14.00
Churn actual: 3.00%
Churn máximo: 3.61%

--- Casuales ---
CLV actual: $55.29
Margen nuevo: $5.20
Churn actual: 6.00%
Churn máximo: 7.86%


,Elasticidad churn,Churn Constantes,¿Cabe Constantes?,Churn Casuales,¿Cabe Casuales?
0,0.5,3.15%,Sí,6.30%,Sí
1,1.0,3.30%,Sí,6.60%,Sí
2,2.5,3.75%,No,7.50%,Sí
